# Initial refactored from R ../initial.py

In [ ]:
import argparse
import time
from datetime import datetime, timedelta, timezone
from typing import List, Tuple

import ccxt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

from dataloader import load_data


CSV_FILE = "data/part2/advanced_orderflow_ws.csv"


def collect_orderflow_data(
    symbol: str = "BTC/USDT",
    poll_interval: int = 5,
    n_iterations: int = 15000,
    csv_file: str = CSV_FILE,
) -> None:
    """
    Collect order book snapshots from BinanceUS using ccxt and write
    microstructure features to a CSV file.

    Parameters
    ----------
    symbol : str
        Trading pair symbol, e.g. 'BTC/USDT'.
    poll_interval : int
        Seconds between API calls.
    n_iterations : int
        Number of snapshots to collect.
    csv_file : str
        Output CSV file path.
    """
    exchange = ccxt.binanceus()

    print(
        f"Starting advanced data collection for {symbol}. "
        f"Polling every {poll_interval}s for {n_iterations} iterations."
    )

    for i in range(1, n_iterations + 1):
        try:
            ob = exchange.fetch_order_book(symbol)

            bids_df = pd.DataFrame(ob["bids"], columns=["price", "amount"])
            asks_df = pd.DataFrame(ob["asks"], columns=["price", "amount"])

            if bids_df.empty or asks_df.empty:
                print(
                    f"[{datetime.now().strftime('%H:%M:%S')}] "
                    "Empty order book, skipping."
                )
                time.sleep(poll_interval)
                continue

            best_bid = bids_df["price"].iloc[0]
            best_ask = asks_df["price"].iloc[0]
            bid_vol_1 = bids_df["amount"].iloc[0]

            ask_vol_1 = asks_df["amount"].iloc[0]

            mid_price = (best_bid + best_ask) / 2
            spread = best_ask - best_bid
            fractional_price = mid_price - int(mid_price)

            micro_price = (best_bid * ask_vol_1 + best_ask * bid_vol_1) / (
                bid_vol_1 + ask_vol_1
            )

            obi_1 = (bid_vol_1 - ask_vol_1) / (bid_vol_1 + ask_vol_1)

            top_bids_10 = bids_df.head(10)
            top_asks_10 = asks_df.head(10)
            bid_vol_10 = top_bids_10["amount"].sum()
            ask_vol_10 = top_asks_10["amount"].sum()
            depth_10 = bid_vol_10 + ask_vol_10
            obi_10 = (bid_vol_10 - ask_vol_10) / (bid_vol_10 + ask_vol_10)

            obi_diff = obi_10 - obi_1
            depth_imbalance_10 = (bid_vol_10 - ask_vol_10) / depth_10 if depth_10 != 0 else 0.0

            current_time = datetime.now()

            row_data = pd.DataFrame(
                [
                    {
                        "timestamp": current_time,
                        "mid_price": mid_price,
                        "micro_price": micro_price,
                        "fractional_price": fractional_price,
                        "spread": spread,
                        "obi_1": obi_1,
                        "obi_10": obi_10,
                        "obi_diff": obi_diff,
                        "bid_vol_10": bid_vol_10,
                        "ask_vol_10": ask_vol_10,
                        "depth_10": depth_10,
                        "depth_imbalance_10": depth_imbalance_10,
                    }
                ]
            )

            header = not pd.io.common.file_exists(csv_file)
            row_data.to_csv(csv_file, mode="a", index=False, header=header)

            print(
                f"[{current_time.strftime('%H:%M:%S')}] "
                f"Iter {i}/{n_iterations} | "
                f"Mid: {mid_price:.2f} | OBI(10): {obi_10:+.2f} | Spread: {spread:.2f}"
            )

        except Exception as e:
            print(
                f"[{datetime.now().strftime('%H:%M:%S')}] "
                f"API error ({type(e).__name__}: {e}). Skipping iteration."
            )

        time.sleep(poll_interval)

    print("Data collection complete.")


def load_and_prepare_data(
    csv_file: str = CSV_FILE,
    horizon: int = 5,
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, List[str]]:
    """
    Compatibility wrapper.

    The project now uses a single shared loader (`dataloader.py`) so that
    training/inference/visualization all see the same feature set and ordering.

    Note: `horizon` is ignored by the shared loader (it mirrors the R script’s
    forward-window logic).
    """
    return load_data(csv_file)


def train_test_split_time_series(
    X: np.ndarray, y: np.ndarray, train_ratio: float = 0.7
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Simple time-based train/test split.
    """
    n = len(y)
    split_idx = int(n * train_ratio)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    return X_train, X_test, y_train, y_test


def run_logistic_regression(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
) -> None:
    """
    Fit and evaluate an L2-regularized logistic regression model.
    """
    if X_train.size == 0 or X_test.size == 0:
        print("Skipping logistic regression: no samples available.")
        return
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        n_jobs=-1,
    )

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    print("=== Logistic Regression (L2) ===")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
    print("\nClassification report:\n", classification_report(y_test, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


def run_random_forest(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
) -> None:
    """
    Fit and evaluate a Random Forest classifier.
    """
    if X_train.size == 0 or X_test.size == 0:
        print("Skipping random forest: no samples available.")
        return

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print("=== Random Forest ===")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
    print("\nClassification report:\n", classification_report(y_test, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


def time_series_cross_validation(
    X_train: np.ndarray, y_train: np.ndarray, n_splits: int = 5
) -> None:
    """
    Time-series cross-validation for logistic regression as an example.
    """
    # If there is only one class in the training data, CV is not meaningful.
    unique_classes = np.unique(y_train)
    if unique_classes.size < 2:
        print(
            "Skipping time-series CV: training data contains only one class "
            f"{unique_classes}."
        )
        return

    tscv = TimeSeriesSplit(n_splits=n_splits)
    scaler = StandardScaler()
    cv_acc: List[float] = []

    print("=== Time Series Cross-Validation (Logistic Regression) ===")
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), 1):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        # Some folds may still have a single class due to small sample size.
        if np.unique(y_tr).size < 2 or np.unique(y_val).size < 2:
            print(f"Fold {fold}: skipped (only one class in this fold).")
            continue

        X_tr_s = scaler.fit_transform(X_tr)
        X_val_s = scaler.transform(X_val)

        model = LogisticRegression(
            penalty="l2", C=1.0, solver="lbfgs", max_iter=1000, n_jobs=-1
        )
        model.fit(X_tr_s, y_tr)
        y_val_pred = model.predict(X_val_s)
        acc = accuracy_score(y_val, y_val_pred)
        cv_acc.append(acc)
        print(f"Fold {fold} accuracy: {acc:.3f}")

    print("Mean CV accuracy:", float(np.mean(cv_acc)))


def main() -> None:
    """
    Entry point: assumes CSV data has already been collected, then
    builds features, trains models, and reports performance.

    If you want to collect fresh data first, call collect_orderflow_data()
    before running the modeling steps.
    """
    p = argparse.ArgumentParser(description="LogReg + RandomForest baselines on order-flow CSV.")
    p.add_argument("--csv", type=str, default=CSV_FILE,
                   help=f"Path to CSV (default: {CSV_FILE})")
    p.add_argument("--train-ratio", type=float, default=0.7)
    p.add_argument("--n-splits", type=int, default=5)
    args = p.parse_args()

    # Uncomment this line if you want to (re)collect data from the exchange:
    # collect_orderflow_data(symbol="BTC/USDT", poll_interval=5, n_iterations=15000)

    df, X, y, feature_cols = load_and_prepare_data(args.csv, horizon=5)
    print(f"Loaded {len(df)} rows with {len(feature_cols)} features.")

    if len(df) == 0 or X.size == 0:
        print(
            "No usable rows after feature/target construction. "
            "You may need to collect more data first."
        )
        return

    X_train, X_test, y_train, y_test = train_test_split_time_series(
        X, y, train_ratio=args.train_ratio
    )
    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

    if X_train.size == 0 or X_test.size == 0:
        print(
            "Train/test split resulted in empty sets. "
            "Collect more data before modeling."
        )
        return

    time_series_cross_validation(X_train, y_train, n_splits=args.n_splits)
    run_logistic_regression(X_train, X_test, y_train, y_test)
    run_random_forest(X_train, X_test, y_train, y_test)


if __name__ == "__main__":
    main()



# Dataloader ../dataloader.py

In [ ]:
"""
Shared data loader (v2) — websocket / event-driven order flow.

Why this file changed (see v1/dataloader.py for the old implementation):
  - The v1 loader was calibrated to a 5-second polling cadence. `shift(24)` was
    assumed to be "~120s backward", `shift(-6)` was "~30s forward". The v2
    websocket capture is event-driven (~4 Hz median, bursty, with multi-second
    gaps) so positional shifts no longer correspond to any fixed time window.
  - v2 resamples each contiguous segment to a regular grid, then uses shifts
    expressed in **grid steps** so every window and label is defined in seconds.
  - v2 segments the stream on large gaps so windows/labels never straddle a
    reconnection or feed outage.
  - v2 drops zero-return rows within a small dead-band so ties aren't silently
    folded into "down".
  - v2 drops `Elapsed_Seconds` (a monotonic feature that extrapolates in val/test).

Defaults below match the knobs we agreed on:
    step_seconds        = 0.5    # 500 ms resample grid
    horizon_seconds     = 15.0
    backward_seconds    = 120.0  # "slow" volatility window
    gap_seconds         = 2.0    # any tick gap > 2s starts a new segment
    deadband_bps        = 0.5    # |future_return| < 0.5 bp → drop row

Multi-scale volatility (for the Vol-Transformer architecture):
    vol_fast_seconds    = 5.0    # short-horizon mid-price std
    vol_med_seconds     = 30.0   # medium-horizon mid-price std
    vol_of_vol_seconds  = 30.0   # rolling std of the fast volatility itself
Plus multi-scale log-returns over {1s, 5s, 15s} to give the model explicit
momentum signals at different horizons.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Public container
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class LoadedData:
    df: pd.DataFrame                 # model-ready, post-resample, post-dropna
    X: np.ndarray                    # (N, F) float64 feature matrix
    y: np.ndarray                    # (N,)   int {0,1} binary up/down label
    feature_cols: List[str]          # column order used to build X
    vol_feature_cols: List[str]      # subset of feature_cols used for FiLM vol conditioning
    segment_ids: np.ndarray          # (N,)   int segment id per row (for seq masking)
    config: dict = field(default_factory=dict)   # effective config used

    @property
    def vol_feature_indices(self) -> List[int]:
        """Indices into feature_cols for the volatility features."""
        return [self.feature_cols.index(c) for c in self.vol_feature_cols]


# ---------------------------------------------------------------------------
# Time parsing (supports MM:SS.s and HH:MM:SS, matching v1 behavior)
# ---------------------------------------------------------------------------

def _parse_time_to_seconds(series: pd.Series) -> np.ndarray:
    """
    Parse a Time column into monotonic elapsed seconds.

    Handles both formats:
      - MM:SS.s   e.g. "11:23.9"  (minute:second.tenths)
      - HH:MM:SS  e.g. "03:49:45"

    Wraps at hour boundaries (a negative diff → add 3600s) to produce a
    monotonic elapsed-seconds vector. NaNs become 0 for the diff.
    """
    s = series.astype(str).str.strip()
    parts = s.str.split(":", expand=True)
    is_hms = s.str.count(":") == 2

    mmss = (
        pd.to_numeric(parts[0], errors="coerce").fillna(0) * 60
        + pd.to_numeric(parts[1], errors="coerce").fillna(0)
    )
    if parts.shape[1] >= 3:
        hms = (
            pd.to_numeric(parts[0], errors="coerce").fillna(0) * 3600
            + pd.to_numeric(parts[1], errors="coerce").fillna(0) * 60
            + pd.to_numeric(parts[2], errors="coerce").fillna(0)
        )
        secs = np.where(is_hms, hms.values, mmss.values)
    else:
        secs = mmss.values

    # Unwrap hour rollovers: whenever the diff goes negative we assume the clock
    # wrapped from e.g. 59:58 → 00:02 and add 3600s to the step.
    steps = np.diff(secs, prepend=secs[0])
    steps = np.where(steps < 0, steps + 3600.0, steps)
    steps[0] = 0.0
    elapsed = np.cumsum(steps)
    return elapsed.astype(float)


# ---------------------------------------------------------------------------
# Column normalization (teacher → internal names)
# ---------------------------------------------------------------------------

def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map: dict[str, str] = {}
    if "Mid Price" in df.columns:
        rename_map["Mid Price"] = "mid_price"
    if "Micro Price" in df.columns:
        rename_map["Micro Price"] = "micro_price"
    if "fractional price" in df.columns:
        rename_map["fractional price"] = "fractional_price"
    if rename_map:
        df = df.rename(columns=rename_map)
    return df


# ---------------------------------------------------------------------------
# Segment detection on gaps
# ---------------------------------------------------------------------------

def _segment_ids_from_gaps(elapsed: np.ndarray, gap_seconds: float) -> np.ndarray:
    """
    Return a per-row segment id such that adjacent rows whose Δt > gap_seconds
    land in different segments.
    """
    if len(elapsed) == 0:
        return np.zeros(0, dtype=np.int64)
    dt = np.diff(elapsed, prepend=elapsed[0])
    breaks = dt > gap_seconds
    # First row starts segment 0; every gap increments the segment id.
    return np.cumsum(breaks).astype(np.int64)


# ---------------------------------------------------------------------------
# Resample a single segment to a fixed grid
# ---------------------------------------------------------------------------

_BASE_COLS = ["mid_price", "micro_price", "fractional_price",
              "spread", "obi_1", "obi_10", "obi_diff"]


def _resample_segment(
    seg: pd.DataFrame,
    step_seconds: float,
    seg_id: int,
) -> pd.DataFrame:
    """
    Resample one contiguous segment to a regular grid.

    Strategy: bin by floor(elapsed / step) and take the **last** tick in each
    bin (the most recent book state at the grid point). Bins with no tick are
    forward-filled from the previous bin **within the same segment only**
    (because inter-segment gaps would mean stale data across a dropout).

    Returns a DataFrame indexed 0..K-1 on the grid. Adds a `segment_id` column.
    """
    if seg.empty:
        return seg.iloc[0:0].copy()

    seg = seg.reset_index(drop=True)
    t = seg["Elapsed_Seconds"].to_numpy()
    t0 = t[0]
    bin_idx = np.floor((t - t0) / step_seconds).astype(np.int64)

    cols = _BASE_COLS
    grid_len = int(bin_idx[-1]) + 1
    grid = pd.DataFrame(
        index=np.arange(grid_len),
        columns=cols,
        dtype=float,
    )

    # "Last tick per bin": groupby bin_idx and take the last observation, which
    # is the most recent book snapshot at that grid point.
    seg_tagged = seg[cols].copy()
    seg_tagged["_bin"] = bin_idx
    last_per_bin = seg_tagged.groupby("_bin", sort=True).last()
    grid.loc[last_per_bin.index, cols] = last_per_bin[cols].to_numpy()

    # Forward-fill empty bins within this segment. Bin 0 always has a tick
    # (it's the first row of the segment) so there should be no leading NaNs.
    grid[cols] = grid[cols].ffill()

    grid["Elapsed_Seconds"] = t0 + grid.index.to_numpy() * step_seconds
    grid["segment_id"] = seg_id
    return grid.reset_index(drop=True)


# ---------------------------------------------------------------------------
# Feature engineering (time-based, per segment)
# ---------------------------------------------------------------------------

def _add_time_of_day_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cyclical encoding of the timestamp modulo 1 hour. Replaces raw Elapsed_Seconds
    as a feature (which was monotonic and extrapolated at val/test time).
    """
    theta = 2.0 * np.pi * (df["Elapsed_Seconds"] % 3600.0) / 3600.0
    df["tod_sin"] = np.sin(theta)
    df["tod_cos"] = np.cos(theta)
    return df


def _per_segment_rolling(
    df: pd.DataFrame,
    step_seconds: float,
    backward_seconds: float,
    horizon_seconds: float,
    vol_fast_seconds: float,
    vol_med_seconds: float,
    vol_of_vol_seconds: float,
    ret_horizons_seconds: Tuple[float, ...],
) -> pd.DataFrame:
    """
    Compute multi-scale volatility + multi-scale return features, the backward
    OBI momentum, and the forward label — all respecting segment boundaries.

    We compute rolling ops on log(mid_price) so the volatility is a ~% std,
    which is scale-free across the dataset.

    Parameters
    ----------
    backward_seconds   : slow volatility window (and OBI momentum lag)
    horizon_seconds    : forward label horizon
    vol_fast_seconds   : short-horizon volatility window
    vol_med_seconds    : medium-horizon volatility window
    vol_of_vol_seconds : rolling std of the *fast* volatility (vol-of-vol)
    ret_horizons_seconds : tuple of backward return windows, one column each
    """
    def to_steps(sec: float) -> int:
        return max(1, int(round(sec / step_seconds)))

    slow_steps = to_steps(backward_seconds)
    fast_steps = to_steps(vol_fast_seconds)
    med_steps = to_steps(vol_med_seconds)
    vov_steps = to_steps(vol_of_vol_seconds)
    fwd_steps = to_steps(horizon_seconds)

    g = df.groupby("segment_id", sort=False, group_keys=False)

    # log-mid is stationary-ish → its rolling std is a fractional volatility
    df["_log_mid"] = np.log(df["mid_price"])

    # --- Multi-scale volatility (std of log-mid over backward windows).
    df["volatility_fast"] = g["_log_mid"].transform(
        lambda s: s.rolling(window=fast_steps, min_periods=fast_steps).std()
    )
    df["volatility_med"] = g["_log_mid"].transform(
        lambda s: s.rolling(window=med_steps, min_periods=med_steps).std()
    )
    df["volatility_slow"] = g["_log_mid"].transform(
        lambda s: s.rolling(window=slow_steps, min_periods=slow_steps).std()
    )

    # Vol-of-vol: std of volatility_fast over the vol_of_vol window.
    df["vol_of_vol"] = df.groupby("segment_id", sort=False, group_keys=False)[
        "volatility_fast"
    ].transform(lambda s: s.rolling(window=vov_steps, min_periods=vov_steps).std())

    # --- Multi-scale backward log-returns.
    for h_sec in ret_horizons_seconds:
        h_steps = to_steps(h_sec)
        col = f"logret_{int(round(h_sec * 1000))}ms"
        df[col] = df.groupby("segment_id", sort=False, group_keys=False)[
            "_log_mid"
        ].transform(lambda s, k=h_steps: s - s.shift(k))

    # OBI momentum over the slow window.
    df["obi_momentum"] = g["obi_10"].transform(lambda s: s - s.shift(slow_steps))

    # Future mid price and label return (per segment).
    df["future_price"] = g["mid_price"].transform(lambda s: s.shift(-fwd_steps))
    df["future_return"] = (df["future_price"] - df["mid_price"]) / df["mid_price"]

    df.drop(columns=["_log_mid"], inplace=True)
    return df


# ---------------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------------

def load_clean_r_style(
    csv_file: str,
    step_seconds: float = 0.5,
    horizon_seconds: float = 15.0,
    backward_seconds: float = 120.0,
    gap_seconds: float = 2.0,
    deadband_bps: float = 0.5,
    vol_fast_seconds: float = 5.0,
    vol_med_seconds: float = 30.0,
    vol_of_vol_seconds: float = 30.0,
    ret_horizons_seconds: Tuple[float, ...] = (1.0, 5.0, 15.0),
) -> LoadedData:
    """
    Load, resample, and label a websocket order-flow CSV.

    Parameters
    ----------
    csv_file : str
        Path to the CSV (old or new schema — columns are normalized).
    step_seconds : float, default 0.5
        Resample grid spacing in seconds.
    horizon_seconds : float, default 15.0
        Forward horizon for the binary label.
    backward_seconds : float, default 120.0
        Backward window for volatility and OBI momentum features.
    gap_seconds : float, default 2.0
        Any tick-to-tick Δt larger than this starts a new segment. Rolling
        windows, shifts, and sequences never cross a segment boundary.
    deadband_bps : float, default 0.5
        Rows whose |future_return| is under this many basis points are treated
        as ties and dropped. 1 bp = 1e-4. Set to 0 to disable.

    Returns
    -------
    LoadedData with:
        df           : model-ready frame (post-resample, post-dropna, post-deadband)
        X            : (N, F) feature matrix
        y            : (N,) int {0,1} label
        feature_cols : list of column names in the order used to build X
        segment_ids  : (N,) segment id per row (use to mask sequence starts)
        config       : dict of effective knobs
    """
    cfg = {
        "step_seconds": step_seconds,
        "horizon_seconds": horizon_seconds,
        "backward_seconds": backward_seconds,
        "gap_seconds": gap_seconds,
        "deadband_bps": deadband_bps,
        "vol_fast_seconds": vol_fast_seconds,
        "vol_med_seconds": vol_med_seconds,
        "vol_of_vol_seconds": vol_of_vol_seconds,
        "ret_horizons_seconds": list(ret_horizons_seconds),
    }

    df = pd.read_csv(csv_file)
    df = _normalize_columns(df)

    missing = [c for c in _BASE_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"CSV missing required columns: {missing}")
    if "Time" not in df.columns:
        raise ValueError("Expected a 'Time' column in the CSV.")

    # 1) Parse Time → monotonic Elapsed_Seconds and segment on gaps.
    df = df.copy()
    df["Elapsed_Seconds"] = _parse_time_to_seconds(df["Time"])
    df["segment_id"] = _segment_ids_from_gaps(
        df["Elapsed_Seconds"].to_numpy(), gap_seconds=gap_seconds
    )

    # 2) Resample each segment to the fixed grid. Re-index segment ids
    #    contiguously on the resampled output.
    resampled_parts: List[pd.DataFrame] = []
    for new_id, (_, seg) in enumerate(df.groupby("segment_id", sort=True)):
        part = _resample_segment(seg, step_seconds=step_seconds, seg_id=new_id)
        if not part.empty:
            resampled_parts.append(part)

    if not resampled_parts:
        return LoadedData(
            df=df.iloc[0:0].copy(),
            X=np.empty((0, 0)),
            y=np.empty((0,), dtype=int),
            feature_cols=[],
            vol_feature_cols=[],
            segment_ids=np.empty((0,), dtype=np.int64),
            config=cfg,
        )

    grid_df = pd.concat(resampled_parts, ignore_index=True)

    # 3) Cyclical time-of-day features (replaces raw Elapsed_Seconds in X).
    grid_df = _add_time_of_day_features(grid_df)

    # 4) Time-based backward windows + forward label (per segment).
    grid_df = _per_segment_rolling(
        grid_df,
        step_seconds=step_seconds,
        backward_seconds=backward_seconds,
        horizon_seconds=horizon_seconds,
        vol_fast_seconds=vol_fast_seconds,
        vol_med_seconds=vol_med_seconds,
        vol_of_vol_seconds=vol_of_vol_seconds,
        ret_horizons_seconds=ret_horizons_seconds,
    )

    # 5) Drop rows with any required NA (backward rolling + future shift edges).
    ret_cols = [
        f"logret_{int(round(h * 1000))}ms" for h in ret_horizons_seconds
    ]
    required = [
        "volatility_fast",
        "volatility_med",
        "volatility_slow",
        "vol_of_vol",
        "obi_momentum",
        "future_price",
        "future_return",
    ] + ret_cols
    df_ready = grid_df.dropna(subset=required).reset_index(drop=True)

    # 6) Apply dead-band to drop ties. 1 bp = 1e-4.
    if deadband_bps > 0:
        thresh = deadband_bps * 1e-4
        keep = df_ready["future_return"].abs() >= thresh
        df_ready = df_ready.loc[keep].reset_index(drop=True)

    # 7) Binary label.
    y = (df_ready["future_return"] > 0).astype(int).to_numpy()

    # 8) Feature columns — explicit allow-list, preserves ordering.
    #    Organized into:
    #      - "price" features (raw book state + cyclical time)
    #      - multi-scale log-return features (explicit momentum signal)
    #      - "vol" features (used both as model input AND as the FiLM conditioner)
    price_cols = [
        "mid_price",
        "micro_price",
        "fractional_price",
        "spread",
        "obi_1",
        "obi_10",
        "obi_diff",
        "tod_sin",
        "tod_cos",
    ]
    vol_feature_cols = [
        "volatility_fast",
        "volatility_med",
        "volatility_slow",
        "vol_of_vol",
        "obi_momentum",
    ]
    feature_cols = price_cols + ret_cols + vol_feature_cols
    X = df_ready[feature_cols].to_numpy(dtype=float)
    segment_ids = df_ready["segment_id"].to_numpy(dtype=np.int64)

    return LoadedData(
        df=df_ready,
        X=X,
        y=y,
        feature_cols=feature_cols,
        vol_feature_cols=vol_feature_cols,
        segment_ids=segment_ids,
        config=cfg,
    )


# ---------------------------------------------------------------------------
# Sequence assembly (respects segment boundaries)
# ---------------------------------------------------------------------------

def valid_sequence_ends(
    segment_ids: np.ndarray,
    seq_len: int,
    stride: int = 1,
    start_offset: int = 0,
) -> np.ndarray:
    """
    Indices `t` such that rows [t - seq_len + 1 .. t] all share the same
    segment_id (i.e. the window doesn't cross a discontinuity).

    Parameters
    ----------
    segment_ids : (N,) array
    seq_len     : window length in grid steps
    stride      : step between consecutive window ends (use >1 for independent
                  test windows)
    start_offset : optional offset applied before striding (used e.g. to align
                   eval strides to the end of a split)

    Returns
    -------
    np.ndarray of end-indices `t` (inclusive) into the feature matrix.
    """
    n = len(segment_ids)
    if n < seq_len:
        return np.empty((0,), dtype=np.int64)
    # A window ending at t is valid iff segment_ids[t - seq_len + 1] == segment_ids[t].
    starts = segment_ids[: n - seq_len + 1]
    ends = segment_ids[seq_len - 1 :]
    mask = starts == ends
    all_valid_ends = np.arange(seq_len - 1, n)[mask]
    if stride <= 1 and start_offset == 0:
        return all_valid_ends
    # Apply offset + stride.
    sel = (np.arange(len(all_valid_ends)) - start_offset) % stride == 0
    sel &= np.arange(len(all_valid_ends)) >= start_offset
    return all_valid_ends[sel]


def build_sequences(
    X: np.ndarray,
    y: np.ndarray,
    segment_ids: np.ndarray,
    seq_len: int,
    stride: int = 1,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Materialize (num_sequences, seq_len, num_features) float32 windows whose
    rows share a segment_id. Label is taken from the window end.

    Uses numpy fancy-indexing with a precomputed valid-end list so we never
    stack a window that straddles a dropout.
    """
    ends = valid_sequence_ends(segment_ids, seq_len=seq_len, stride=stride)
    if ends.size == 0:
        return np.empty((0, seq_len, X.shape[1]), dtype=np.float32), np.empty((0,), dtype=np.float32)
    # Vectorized gather: for each end t, take rows [t-seq_len+1 .. t].
    offsets = np.arange(-seq_len + 1, 1)  # (seq_len,), inclusive of 0
    idx = ends[:, None] + offsets[None, :]  # (num_seq, seq_len)
    X_seq = X[idx].astype(np.float32, copy=False)
    y_seq = y[ends].astype(np.float32, copy=False)
    return X_seq, y_seq


# ---------------------------------------------------------------------------
# Backward-compatible wrapper
# ---------------------------------------------------------------------------

def load_data(
    csv_file: str,
    **kwargs,
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, List[str]]:
    """Tuple return kept for legacy callers (XGBoost baseline, initial.py)."""
    out = load_clean_r_style(csv_file, **kwargs)
    return out.df, out.X, out.y, out.feature_cols


# Transfromer Training ../transformer.py

In [ ]:
"""
Bitcoin Up/Down Vol-Transformer Classifier (v2)
===============================================

Architecture upgrades over v1 (see v1/transformer.py for the old model):
  - Time-based data pipeline (via dataloader.py v2). Horizon / windows / segments
    are expressed in seconds, not tick counts.
  - Causal 1D conv stem before the transformer, to capture short-range tick
    dynamics cheaply.
  - Dual-stream input projection: price features and volatility features are
    projected separately and summed into the model dim (so the optimizer can
    balance their magnitudes independently).
  - Per-layer FiLM conditioning: each transformer layer is modulated by (γ, β)
    vectors predicted from a pooled volatility summary of the current window,
    giving the model an explicit "current vol regime" knob.
  - Attention pooling with a learned query (instead of last-token readout),
    so transient signals anywhere in the window can drive the prediction.
  - Segment-aware sequence construction: windows never straddle a websocket
    dropout (see dataloader.valid_sequence_ends).

Infra upgrades:
  - Versioned outputs/vN/, scaler + config.json + metrics saved per run.
  - Train loader shuffles; eval loaders don't. Test sequences use stride to
    produce (mostly) independent samples for honest metrics.
  - float32 throughout; pin_memory when on GPU.
  - cudnn determinism when a seed is passed.
  - Threshold sweep over full [0.01, 0.99].
  - Config dump is complete: model + data + training knobs.
"""

from __future__ import annotations

import argparse
import json
import os
import pickle
import random
import time
from pathlib import Path
from typing import List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

from dataloader import (
    LoadedData,
    load_clean_r_style,
    valid_sequence_ends,
    build_sequences,
)


# ---------------------------------------------------------------------------
# Output versioning
# ---------------------------------------------------------------------------

def get_next_version_dir(base: str = "outputs") -> Path:
    base_path = Path(base)
    base_path.mkdir(exist_ok=True)
    existing = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith("v")]
    versions = []
    for d in existing:
        try:
            versions.append(int(d.name[1:]))
        except ValueError:
            pass
    next_v = max(versions, default=0) + 1
    out_dir = base_path / f"v{next_v}"
    out_dir.mkdir()
    print(f"Output directory: {out_dir}")
    return out_dir


# ---------------------------------------------------------------------------
# Positional encoding
# ---------------------------------------------------------------------------

class PositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding (Vaswani et al.)."""

    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class LearnablePositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1) -> None:
        super().__init__()
        self.pe = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


# ---------------------------------------------------------------------------
# Causal 1D conv stem
# ---------------------------------------------------------------------------

class CausalConv1d(nn.Module):
    """1D convolution with left-padding so output at time t only depends on t'≤t."""

    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, dilation: int = 1):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, dilation=dilation)

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # x: (B, C, T)
        x = nn.functional.pad(x, (self.pad, 0))
        return self.conv(x)


class ConvStem(nn.Module):
    """Two dilated causal convs with a residual; runs over the time dim."""

    def __init__(self, d_model: int, kernel_size: int = 3, dropout: float = 0.1):
        super().__init__()
        self.c1 = CausalConv1d(d_model, d_model, kernel_size, dilation=1)
        self.c2 = CausalConv1d(d_model, d_model, kernel_size, dilation=2)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # x: (B, T, D)
        h = x.transpose(1, 2)  # (B, D, T)
        h = self.act(self.c1(h))
        h = self.drop(h)
        h = self.c2(h)
        h = h.transpose(1, 2)  # (B, T, D)
        return self.norm(x + h)


# ---------------------------------------------------------------------------
# Volatility summary → per-layer FiLM parameters
# ---------------------------------------------------------------------------

class VolFiLM(nn.Module):
    """
    Produce (γ, β) per transformer layer from a pooled volatility summary.

    Input to forward:  x_vol of shape (B, T, V) where V is len(vol_feature_cols)
    Output:            gammas, betas each shape (num_layers, B, 1, d_model)
    """

    def __init__(
        self,
        vol_dim: int,
        d_model: int,
        num_layers: int,
        hidden: int = 64,
        dropout: float = 0.1,
        pool: str = "mean",
    ) -> None:
        super().__init__()
        self.num_layers = num_layers
        self.d_model = d_model
        self.pool = pool
        self.mlp = nn.Sequential(
            nn.Linear(vol_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_layers * d_model * 2),
        )
        # Zero-init the last layer so FiLM starts as identity (γ=1, β=0).
        last = self.mlp[-1]
        nn.init.zeros_(last.weight)
        nn.init.zeros_(last.bias)

    def forward(self, x_vol: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # Pool over the time dim. Mean is a defensible default; could try attn later.
        if self.pool == "last":
            summary = x_vol[:, -1, :]
        else:
            summary = x_vol.mean(dim=1)
        params = self.mlp(summary)  # (B, num_layers * d_model * 2)
        B = params.size(0)
        params = params.view(B, self.num_layers, 2, self.d_model)
        # Identity at init: γ = 1 + Δγ, β = Δβ.
        gamma = 1.0 + params[:, :, 0, :]  # (B, L, D)
        beta = params[:, :, 1, :]          # (B, L, D)
        # Reshape to (L, B, 1, D) for per-layer indexing and broadcasting over T.
        gamma = gamma.permute(1, 0, 2).unsqueeze(2).contiguous()
        beta = beta.permute(1, 0, 2).unsqueeze(2).contiguous()
        return gamma, beta


# ---------------------------------------------------------------------------
# Transformer layer with FiLM + optional diagonal attention bias
# ---------------------------------------------------------------------------

class FiLMTransformerLayer(nn.Module):
    def __init__(
        self,
        d_model: int,
        nhead: int,
        dim_feedforward: int,
        dropout: float,
        attn_diagonal_bias: float = 0.0,
    ) -> None:
        super().__init__()
        self.attn_diagonal_bias = attn_diagonal_bias
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )

    def _diag_bias(self, T: int, device: torch.device) -> Optional[torch.Tensor]:
        if self.attn_diagonal_bias <= 0:
            return None
        i = torch.arange(T, device=device, dtype=torch.float32)
        j = torch.arange(T, device=device, dtype=torch.float32)
        dist = (i.unsqueeze(1) - j.unsqueeze(0)).abs()
        return -self.attn_diagonal_bias * dist

    def forward(
        self,
        x: torch.Tensor,
        gamma: torch.Tensor,  # (B, 1, D)
        beta: torch.Tensor,   # (B, 1, D)
        return_attn: bool = False,
    ):
        # Pre-norm, then FiLM, then self-attention.
        x_norm = self.norm1(x)
        x_cond = gamma * x_norm + beta
        attn_out, attn_w = self.self_attn(
            x_cond, x_cond, x_cond,
            attn_mask=self._diag_bias(x.size(1), x.device),
            need_weights=return_attn,
            average_attn_weights=False,
        )
        x = x + self.dropout1(attn_out)
        x = x + self.dropout2(self.ff(self.norm2(x)))
        if return_attn:
            return x, attn_w
        return x


# ---------------------------------------------------------------------------
# Attention pooling (learned query)
# ---------------------------------------------------------------------------

class AttentionPool(nn.Module):
    def __init__(self, d_model: int, nhead: int = 4, dropout: float = 0.1) -> None:
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.size(0)
        q = self.query.expand(B, -1, -1)            # (B, 1, D)
        out, _ = self.attn(q, x, x)                 # (B, 1, D)
        return self.norm(out.squeeze(1))            # (B, D)


# ---------------------------------------------------------------------------
# Full model
# ---------------------------------------------------------------------------

class VolTransformer(nn.Module):
    def __init__(
        self,
        input_dim: int,
        vol_indices: List[int],
        d_model: int = 96,
        nhead: int = 6,
        num_layers: int = 3,
        dim_feedforward: int = 384,
        dropout: float = 0.2,
        pos_encoding: str = "sinusoidal",
        attn_diagonal_bias: float = 0.0,
        conv_kernel: int = 3,
    ) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.vol_indices = list(vol_indices)
        self.price_indices = [i for i in range(input_dim) if i not in set(self.vol_indices)]

        # Dual-stream projections (sum to d_model so each stream has full bandwidth).
        self.price_proj = nn.Linear(len(self.price_indices), d_model)
        self.vol_proj = nn.Linear(len(self.vol_indices), d_model)

        if pos_encoding == "sinusoidal":
            self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        elif pos_encoding == "learnable":
            self.pos_enc = LearnablePositionalEncoding(d_model, dropout=dropout)
        else:
            raise ValueError(f"pos_encoding must be 'sinusoidal' or 'learnable', got {pos_encoding!r}")

        self.conv_stem = ConvStem(d_model, kernel_size=conv_kernel, dropout=dropout)
        self.film = VolFiLM(
            vol_dim=len(self.vol_indices),
            d_model=d_model,
            num_layers=num_layers,
            dropout=dropout,
        )
        self.layers = nn.ModuleList(
            [
                FiLMTransformerLayer(
                    d_model=d_model,
                    nhead=nhead,
                    dim_feedforward=dim_feedforward,
                    dropout=dropout,
                    attn_diagonal_bias=attn_diagonal_bias,
                )
                for _ in range(num_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.pool = AttentionPool(d_model, nhead=min(nhead, 4), dropout=dropout)
        self.cls_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )

    def _split(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x_price = x[..., self.price_indices]
        x_vol = x[..., self.vol_indices]
        return x_price, x_vol

    def _encode(self, x: torch.Tensor, return_attn: bool = False):
        """Run the model up to the pooled representation (just before cls_head)."""
        x_price, x_vol = self._split(x)
        h = self.price_proj(x_price) + self.vol_proj(x_vol)
        h = self.pos_enc(h)
        h = self.conv_stem(h)

        gammas, betas = self.film(x_vol)  # (L, B, 1, D)
        last_attn = None
        for i, layer in enumerate(self.layers):
            if return_attn and i == len(self.layers) - 1:
                h, last_attn = layer(h, gammas[i], betas[i], return_attn=True)
            else:
                h = layer(h, gammas[i], betas[i], return_attn=False)

        h = self.final_norm(h)
        pooled = self.pool(h)
        return pooled, last_attn

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Return (B, d_model) pooled embeddings — handy for t-SNE / probes."""
        pooled, _ = self._encode(x, return_attn=False)
        return pooled

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        pooled, last_attn = self._encode(x, return_attn=return_attn)
        logits = self.cls_head(pooled).squeeze(-1)
        if return_attn:
            return logits, last_attn
        return logits


# ---------------------------------------------------------------------------
# Plot helpers
# ---------------------------------------------------------------------------

def save_tsne_plot(
    model: "VolTransformer",
    loader: DataLoader,
    device: torch.device,
    out_path: Path,
    n_samples: int = 1500,
    perplexity: int = 30,
    title: str = "t-SNE of Vol-Transformer pooled embeddings",
) -> None:
    """
    Project the pooled (pre-head) embeddings into 2D with t-SNE, save a side-
    by-side scatter coloured by true label and by model confidence.

    Skips silently if scikit-learn is missing or the loader yields no samples.
    """
    try:
        from sklearn.manifold import TSNE
    except ImportError:
        print("  Skipping t-SNE: sklearn.manifold.TSNE unavailable.")
        return

    model.eval()
    feats_list, y_list, logits_list = [], [], []
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb_d = xb.to(device)
            pooled = model.encode(xb_d)
            logits = model.cls_head(pooled).squeeze(-1)
            feats_list.append(pooled.cpu().numpy())
            logits_list.append(logits.cpu().numpy())
            y_list.append(yb.numpy())
            total += xb.size(0)
            if total >= n_samples:
                break

    if total == 0:
        print("  Skipping t-SNE: no samples in loader.")
        return

    feats = np.concatenate(feats_list)[:n_samples]
    y = np.concatenate(y_list)[:n_samples]
    logits = np.concatenate(logits_list)[:n_samples]
    probs = 1.0 / (1.0 + np.exp(-logits))

    # t-SNE is O(n²) in memory; keep perplexity valid for tiny samples too.
    eff_perp = max(5, min(perplexity, (len(feats) - 1) // 3))
    print(f"  Running t-SNE on {len(feats)} samples (perplexity={eff_perp}) …")
    tsne = TSNE(n_components=2, perplexity=eff_perp, random_state=42, init="pca")
    X2 = tsne.fit_transform(feats)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    ax = axes[0]
    sc = ax.scatter(X2[:, 0], X2[:, 1], c=y, cmap="coolwarm",
                    alpha=0.7, s=18, linewidths=0)
    plt.colorbar(sc, ax=ax, label="True label (0=down, 1=up)")
    ax.set_title("Coloured by True Label")
    ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2"); ax.grid(True, alpha=0.2)

    ax = axes[1]
    sc2 = ax.scatter(X2[:, 0], X2[:, 1], c=probs, cmap="RdYlGn",
                     alpha=0.7, s=18, linewidths=0, vmin=0, vmax=1)
    plt.colorbar(sc2, ax=ax, label="Model confidence P(up)")
    ax.set_title("Coloured by Model Confidence")
    ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2"); ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig(out_path, format="png", bbox_inches="tight", dpi=150)
    plt.close()
    print(f"  Saved t-SNE plot → {out_path}")


def save_attention_heatmap(attn: torch.Tensor, out_path: Path, sample_idx: int = 0, head: int = 0) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    a = attn.detach().cpu().numpy()
    if a.ndim != 4:
        raise ValueError(f"Expected (B,H,T,T), got {a.shape}")
    plt.figure(figsize=(7, 6))
    sns.heatmap(a[sample_idx, head], cmap="viridis")
    plt.title("Self-attention (last layer)")
    plt.xlabel("Key timestep"); plt.ylabel("Query timestep")
    plt.tight_layout(); plt.savefig(out_path, dpi=200); plt.close()


def save_training_curves(history: dict, out_dir: Path) -> None:
    epochs = range(1, len(history["train_loss"]) + 1)
    best_ep = history.get("best_epoch_auc")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Training History", fontsize=14, fontweight="bold")
    ax = axes[0]
    ax.plot(epochs, history["train_loss"], label="Train Loss", linewidth=2)
    ax.plot(epochs, history["val_loss"], label="Val Loss", linewidth=2, linestyle="--")
    if best_ep: ax.axvline(best_ep, color="red", linestyle=":", alpha=0.6, label=f"Best ({best_ep})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("BCE Loss"); ax.set_title("Loss"); ax.legend(); ax.grid(True, alpha=0.3)
    ax = axes[1]
    ax.plot(epochs, history["train_auc"], label="Train AUC", linewidth=2)
    ax.plot(epochs, history["val_auc"], label="Val AUC", linewidth=2, linestyle="--")
    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5, label="Random")
    if best_ep: ax.axvline(best_ep, color="red", linestyle=":", alpha=0.6, label=f"Best ({best_ep})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("ROC-AUC"); ax.set_title("AUC"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    p = out_dir / "training_curves.png"
    plt.savefig(p, format="png", bbox_inches="tight", dpi=150); plt.close()
    print(f"  Saved training curves → {p}")


# ---------------------------------------------------------------------------
# Training helpers
# ---------------------------------------------------------------------------

def compute_pos_weight(y: np.ndarray, device: torch.device) -> torch.Tensor:
    n_pos = float(y.sum()); n_neg = float(len(y) - n_pos)
    ratio = n_neg / max(n_pos, 1)
    print(f"  Class balance — pos: {int(n_pos)}, neg: {int(n_neg)}, pos_weight: {ratio:.3f}")
    return torch.tensor([ratio], dtype=torch.float, device=device)


def find_best_threshold(logits: np.ndarray, targets: np.ndarray) -> float:
    """Sweep [0.01, 0.99] and pick threshold with best macro F1."""
    probs = 1.0 / (1.0 + np.exp(-logits))
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.01, 1.0, 0.01):
        preds = (probs >= t).astype(int)
        f1 = f1_score(targets, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t


def collect_logits(model, loader, device) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_logits, all_t = [], []
    with torch.no_grad():
        for xb, yb in loader:
            all_logits.append(model(xb.to(device)).cpu().numpy())
            all_t.append(yb.numpy())
    return np.concatenate(all_logits), np.concatenate(all_t)


def eval_auc_from(logits: np.ndarray, targets: np.ndarray) -> float:
    try:
        return roc_auc_score(targets, 1.0 / (1.0 + np.exp(-logits)))
    except ValueError:
        return 0.5


# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------

def train_transformer(
    csv_file: str = "data/part2/advanced_orderflow_ws.csv",
    # Loader knobs (time units)
    step_seconds: float = 0.5,
    horizon_seconds: float = 15.0,
    backward_seconds: float = 120.0,
    gap_seconds: float = 2.0,
    deadband_bps: float = 0.5,
    vol_fast_seconds: float = 5.0,
    vol_med_seconds: float = 30.0,
    vol_of_vol_seconds: float = 30.0,
    # Sequence shape
    seq_len: int = 30,
    test_stride: Optional[int] = None,   # None → horizon_seconds/step_seconds (independent windows)
    # Training
    n_epochs: int = 50,
    batch_size: int = 128,
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    patience: int = 8,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    outputs_base: str = "outputs",
    seed: Optional[int] = None,
    # Model
    d_model: int = 96,
    nhead: int = 6,
    num_layers: int = 3,
    dim_feedforward: int = 384,
    dropout: float = 0.2,
    pos_encoding: str = "sinusoidal",
    attn_diagonal_bias: float = 0.0,
    conv_kernel: int = 3,
) -> None:

    if seed is None:
        seed = int(time.time() * 1e6) % (2**31)
        print(f"Using random seed: {seed}")
    else:
        print(f"Using fixed seed: {seed}")
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    out_dir = get_next_version_dir(outputs_base)
    checkpoint_path = out_dir / "transformer_best.pt"
    scaler_path = out_dir / "scaler.pkl"

    # 1. Load & clean (time-based, segment-aware).
    loaded: LoadedData = load_clean_r_style(
        csv_file,
        step_seconds=step_seconds,
        horizon_seconds=horizon_seconds,
        backward_seconds=backward_seconds,
        gap_seconds=gap_seconds,
        deadband_bps=deadband_bps,
        vol_fast_seconds=vol_fast_seconds,
        vol_med_seconds=vol_med_seconds,
        vol_of_vol_seconds=vol_of_vol_seconds,
    )
    print(f"Loaded {len(loaded.df)} rows, {len(loaded.feature_cols)} features, "
          f"{len(set(loaded.segment_ids))} segments.")
    print(f"  price features : {[c for c in loaded.feature_cols if c not in loaded.vol_feature_cols]}")
    print(f"  vol features   : {loaded.vol_feature_cols}")
    if loaded.X.size == 0:
        print("No usable rows. Aborting."); return

    # 2. Temporal split (on rows, not sequences — sequences are rebuilt per split).
    n = len(loaded.X)
    n_test = int(n * test_ratio)
    n_val = int(n * val_ratio)
    n_train = n - n_val - n_test

    X_train = loaded.X[:n_train]; y_train = loaded.y[:n_train]; seg_train = loaded.segment_ids[:n_train]
    X_val   = loaded.X[n_train:n_train + n_val]; y_val = loaded.y[n_train:n_train + n_val]; seg_val = loaded.segment_ids[n_train:n_train + n_val]
    X_test  = loaded.X[n_train + n_val:]; y_test = loaded.y[n_train + n_val:]; seg_test = loaded.segment_ids[n_train + n_val:]
    print(f"Split — train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")

    # 3. Scaler fit on TRAIN only (standard for time series).
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)
    with open(scaler_path, "wb") as f:
        pickle.dump({"scaler": scaler, "feature_cols": loaded.feature_cols,
                     "vol_feature_cols": loaded.vol_feature_cols}, f)
    print(f"  Saved scaler → {scaler_path}")

    # 4. Build segment-aware sequences. Train uses stride 1; test uses horizon stride
    #    so independent windows don't over-count correlated predictions in metrics.
    if test_stride is None:
        test_stride = max(1, int(round(horizon_seconds / step_seconds)))
    Xtr, ytr = build_sequences(X_train_s, y_train, seg_train, seq_len=seq_len, stride=1)
    Xvl, yvl = build_sequences(X_val_s, y_val, seg_val, seq_len=seq_len, stride=1)
    Xte, yte = build_sequences(X_test_s, y_test, seg_test, seq_len=seq_len, stride=test_stride)
    print(f"Sequences — train: {Xtr.shape}, val: {Xvl.shape}, test (stride={test_stride}): {Xte.shape}")

    if Xtr.shape[0] == 0 or Xvl.shape[0] == 0 or Xte.shape[0] == 0:
        print("Empty sequence set after segment filter; try a shorter seq_len or a larger gap_seconds.")
        return

    # 5. Loaders.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    pin = device.type == "cuda"

    def to_tensor(x): return torch.from_numpy(x)
    train_ds = TensorDataset(to_tensor(Xtr), to_tensor(ytr))
    val_ds = TensorDataset(to_tensor(Xvl), to_tensor(yvl))
    test_ds = TensorDataset(to_tensor(Xte), to_tensor(yte))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=pin, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=pin)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=pin)

    # 6. Model / loss / optimizer.
    model = VolTransformer(
        input_dim=loaded.X.shape[1],
        vol_indices=loaded.vol_feature_indices,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout,
        pos_encoding=pos_encoding,
        attn_diagonal_bias=attn_diagonal_bias,
        conv_kernel=conv_kernel,
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")

    pos_weight = compute_pos_weight(ytr, device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3
    )

    # 7. Training — checkpoint by val AUC.
    print("\n=== Vol-Transformer — Bitcoin Up/Down ===")
    history = {"train_loss": [], "val_loss": [], "train_auc": [], "val_auc": [], "best_epoch_auc": None}
    best_val_auc = 0.0
    epochs_no_improve = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=pin)
            yb = yb.to(device, non_blocking=pin)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)
        epoch_loss /= len(train_loader.dataset)

        # Val: collect logits once, reuse for loss and AUC.
        val_logits, val_targets = collect_logits(model, val_loader, device)
        # Numerically stable pos-weighted BCE from logits+targets:
        #   target=1: w * log(1 + exp(-logit)) = w * logaddexp(0, -logit)
        #   target=0:      log(1 + exp( logit)) =     logaddexp(0,  logit)
        w = float(pos_weight.item())
        pos_mask = val_targets > 0.5
        per_sample = np.where(
            pos_mask,
            w * np.logaddexp(0.0, -val_logits),
            np.logaddexp(0.0, val_logits),
        )
        val_loss = float(per_sample.mean())
        val_auc = eval_auc_from(val_logits, val_targets)
        train_logits, train_targets = collect_logits(model, train_loader, device)
        train_auc = eval_auc_from(train_logits, train_targets)

        scheduler.step(val_auc)
        cur_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(epoch_loss)
        history["val_loss"].append(val_loss)
        history["train_auc"].append(train_auc)
        history["val_auc"].append(val_auc)

        print(f"Epoch {epoch:3d}/{n_epochs} | loss {epoch_loss:.4f}/{val_loss:.4f} | "
              f"AUC {train_auc:.4f}/{val_auc:.4f} | lr {cur_lr:.2e}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            epochs_no_improve = 0
            history["best_epoch_auc"] = epoch
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  ✓ Checkpoint saved (val AUC={val_auc:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}."); break

    save_training_curves(history, out_dir)

    # 8. Final test evaluation (best checkpoint, threshold tuned on val).
    try:
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print(f"\nLoaded best checkpoint (epoch {history['best_epoch_auc']}).")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")

    test_logits, test_targets = collect_logits(model, test_loader, device)
    val_logits, val_targets = collect_logits(model, val_loader, device)
    best_threshold = find_best_threshold(val_logits, val_targets)
    print(f"Best threshold (val): {best_threshold:.2f}")

    y_prob = 1.0 / (1.0 + np.exp(-test_logits))
    y_pred = (y_prob >= best_threshold).astype(int)
    acc = accuracy_score(test_targets, y_pred)
    try:
        auc = roc_auc_score(test_targets, y_prob)
    except ValueError:
        auc = float("nan")

    print(f"\nTest Accuracy : {acc:.4f}")
    print(f"Test ROC-AUC  : {auc:.4f}")
    print("\nClassification Report:\n", classification_report(test_targets, y_pred, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(test_targets, y_pred))

    # 9. Persist everything useful for inference + reproduction.
    config = {
        "csv_file": csv_file,
        "data": loaded.config,
        "seq_len": seq_len,
        "test_stride": test_stride,
        "val_ratio": val_ratio, "test_ratio": test_ratio,
        "feature_cols": loaded.feature_cols,
        "vol_feature_cols": loaded.vol_feature_cols,
        "model": {
            "d_model": d_model, "nhead": nhead, "num_layers": num_layers,
            "dim_feedforward": dim_feedforward, "dropout": dropout,
            "pos_encoding": pos_encoding, "attn_diagonal_bias": attn_diagonal_bias,
            "conv_kernel": conv_kernel,
        },
        "training": {
            "n_epochs": n_epochs, "batch_size": batch_size, "lr": lr,
            "weight_decay": weight_decay, "patience": patience,
            "seed": seed,
        },
        "results": {
            "best_epoch_auc": history["best_epoch_auc"],
            "best_val_auc": best_val_auc,
            "test_accuracy": float(acc),
            "test_auc": float(auc) if not np.isnan(auc) else None,
            "threshold": best_threshold,
            "n_params": n_params,
        },
    }
    with open(out_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    with open(out_dir / "metrics_summary.txt", "w") as f:
        f.write(f"Version dir      : {out_dir}\n")
        f.write(f"CSV              : {csv_file}\n")
        f.write(f"Horizon (s)      : {horizon_seconds}\n")
        f.write(f"Step (s)         : {step_seconds}\n")
        f.write(f"Seq len (steps)  : {seq_len}  (~ {seq_len * step_seconds:.1f}s context)\n")
        f.write(f"Seed             : {seed}\n")
        f.write(f"Best epoch (AUC) : {history['best_epoch_auc']}\n")
        f.write(f"Best val AUC     : {best_val_auc:.4f}\n")
        f.write(f"Threshold        : {best_threshold:.2f}\n")
        f.write(f"Test Accuracy    : {acc:.4f}\n")
        f.write(f"Test AUC         : {auc:.4f}\n\n")
        f.write(classification_report(test_targets, y_pred, zero_division=0))
    print(f"  Saved metrics summary → {out_dir / 'metrics_summary.txt'}")

    # 10. Self-attention heatmap from a val sample.
    try:
        xb0, _ = next(iter(val_loader))
        _, attn = model(xb0.to(device), return_attn=True)
        if attn is not None:
            save_attention_heatmap(attn, out_dir / "attention_heatmap.png")
            np.save(out_dir / "attention_weights.npy", attn.detach().cpu().numpy())
            print(f"  Saved attention heatmap → {out_dir / 'attention_heatmap.png'}")
    except Exception as e:
        print(f"  Could not save attention heatmap: {e}")

    # 11. t-SNE of pooled embeddings on the val set.
    try:
        save_tsne_plot(
            model=model,
            loader=val_loader,
            device=device,
            out_path=out_dir / "tsne.png",
            n_samples=1500,
            perplexity=30,
            title=f"t-SNE of pooled embeddings  ({out_dir.name}, val set)",
        )
    except Exception as e:
        print(f"  Could not save t-SNE plot: {e}")

    print(f"\nAll outputs saved to: {out_dir}/")


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Bitcoin Up/Down Vol-Transformer")
    p.add_argument("--csv", type=str, default="data/part2/advanced_orderflow_ws.csv")
    # Loader
    p.add_argument("--step-seconds", type=float, default=0.5)
    p.add_argument("--horizon-seconds", type=float, default=15.0)
    p.add_argument("--backward-seconds", type=float, default=120.0)
    p.add_argument("--gap-seconds", type=float, default=2.0)
    p.add_argument("--deadband-bps", type=float, default=0.5)
    p.add_argument("--vol-fast-seconds", type=float, default=5.0)
    p.add_argument("--vol-med-seconds", type=float, default=30.0)
    p.add_argument("--vol-of-vol-seconds", type=float, default=30.0)
    # Sequence
    p.add_argument("--seq-len", type=int, default=30, help="window length in grid steps")
    p.add_argument("--test-stride", type=int, default=None,
                   help="stride for test windows; default = horizon/step (independent).")
    # Training
    p.add_argument("--epochs", type=int, default=50)
    p.add_argument("--batch-size", type=int, default=128)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--weight-decay", type=float, default=1e-4)
    p.add_argument("--patience", type=int, default=8)
    p.add_argument("--val-ratio", type=float, default=0.15)
    p.add_argument("--test-ratio", type=float, default=0.15)
    p.add_argument("--outputs-base", type=str, default="outputs")
    p.add_argument("--seed", type=int, default=None)
    # Model
    p.add_argument("--d-model", type=int, default=96)
    p.add_argument("--nhead", type=int, default=6)
    p.add_argument("--num-layers", type=int, default=3)
    p.add_argument("--dim-feedforward", type=int, default=384)
    p.add_argument("--dropout", type=float, default=0.2)
    p.add_argument("--pos-encoding", type=str, default="sinusoidal",
                   choices=["sinusoidal", "learnable"])
    p.add_argument("--attn-diagonal-bias", type=float, default=0.0)
    p.add_argument("--conv-kernel", type=int, default=3)
    return p.parse_args()


if __name__ == "__main__":
    a = parse_args()
    train_transformer(
        csv_file=a.csv,
        step_seconds=a.step_seconds, horizon_seconds=a.horizon_seconds,
        backward_seconds=a.backward_seconds, gap_seconds=a.gap_seconds,
        deadband_bps=a.deadband_bps,
        vol_fast_seconds=a.vol_fast_seconds, vol_med_seconds=a.vol_med_seconds,
        vol_of_vol_seconds=a.vol_of_vol_seconds,
        seq_len=a.seq_len, test_stride=a.test_stride,
        n_epochs=a.epochs, batch_size=a.batch_size, lr=a.lr,
        weight_decay=a.weight_decay, patience=a.patience,
        val_ratio=a.val_ratio, test_ratio=a.test_ratio,
        outputs_base=a.outputs_base, seed=a.seed,
        d_model=a.d_model, nhead=a.nhead, num_layers=a.num_layers,
        dim_feedforward=a.dim_feedforward, dropout=a.dropout,
        pos_encoding=a.pos_encoding, attn_diagonal_bias=a.attn_diagonal_bias,
        conv_kernel=a.conv_kernel,
    )


# TSNE ../tse.py

In [ ]:
"""
Standalone t-SNE visualisation of Vol-Transformer pooled embeddings.

Rebuilds the val/test windows exactly like training did, loads the best
checkpoint, runs the model up to the pooled representation, and saves a
tsne.png into the run folder. Useful when you want to regenerate the t-SNE
with different knobs (perplexity, n_samples, val vs test) without retraining.

Usage:
    python tsne.py                           # latest outputs/vN/, val set
    python tsne.py --version 7               # use outputs/v7/
    python tsne.py --split test              # plot test windows instead
    python tsne.py --n-samples 3000 --perplexity 50
"""

from __future__ import annotations

import argparse
import json
import pickle
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from dataloader import load_clean_r_style, build_sequences
from transformer import VolTransformer, save_tsne_plot


def find_latest_version(base: str = "outputs") -> Path:
    base_path = Path(base)
    dirs = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith("v")]
    if not dirs:
        raise FileNotFoundError(f"No versioned run folders found in '{base}'.")
    versions = []
    for d in dirs:
        try:
            versions.append((int(d.name[1:]), d))
        except ValueError:
            pass
    versions.sort(key=lambda x: x[0])
    return versions[-1][1]


def rebuild_split_loader(
    cfg: dict,
    split: str,
    scaler_dict: dict,
    batch_size: int,
):
    """Recreate val or test DataLoader using the exact config of the saved run."""
    data_cfg = cfg["data"]
    loaded = load_clean_r_style(
        cfg["csv_file"],
        step_seconds=data_cfg["step_seconds"],
        horizon_seconds=data_cfg["horizon_seconds"],
        backward_seconds=data_cfg["backward_seconds"],
        gap_seconds=data_cfg["gap_seconds"],
        deadband_bps=data_cfg["deadband_bps"],
        vol_fast_seconds=data_cfg.get("vol_fast_seconds", 5.0),
        vol_med_seconds=data_cfg.get("vol_med_seconds", 30.0),
        vol_of_vol_seconds=data_cfg.get("vol_of_vol_seconds", 30.0),
    )

    n = len(loaded.X)
    n_test = int(n * cfg["test_ratio"])
    n_val = int(n * cfg["val_ratio"])
    n_train = n - n_val - n_test

    if split == "val":
        sl = slice(n_train, n_train + n_val)
        stride = 1
    elif split == "test":
        sl = slice(n_train + n_val, None)
        stride = cfg.get("test_stride") or max(
            1, int(round(data_cfg["horizon_seconds"] / data_cfg["step_seconds"]))
        )
    else:
        raise ValueError(f"split must be 'val' or 'test', got {split!r}")

    X = loaded.X[sl]; y = loaded.y[sl]; seg = loaded.segment_ids[sl]
    X = scaler_dict["scaler"].transform(X)
    Xs, ys = build_sequences(X, y, seg, seq_len=cfg["seq_len"], stride=stride)
    ds = TensorDataset(torch.from_numpy(Xs), torch.from_numpy(ys))
    return DataLoader(ds, batch_size=batch_size, shuffle=False), loaded


def main() -> None:
    p = argparse.ArgumentParser(description="t-SNE of Vol-Transformer pooled embeddings")
    p.add_argument("--outputs-base", type=str, default="outputs")
    p.add_argument("--version", type=int, default=None, help="Run version (default: latest)")
    p.add_argument("--split", type=str, default="val", choices=["val", "test"])
    p.add_argument("--n-samples", type=int, default=1500)
    p.add_argument("--perplexity", type=int, default=30)
    p.add_argument("--batch-size", type=int, default=256)
    args = p.parse_args()

    run_dir = (Path(args.outputs_base) / f"v{args.version}"
               if args.version is not None else find_latest_version(args.outputs_base))
    print(f"Loading from: {run_dir}")

    ckpt = run_dir / "transformer_best.pt"
    scaler_pkl = run_dir / "scaler.pkl"
    config_path = run_dir / "config.json"
    for p_ in (ckpt, scaler_pkl, config_path):
        if not p_.exists():
            raise FileNotFoundError(f"Missing: {p_}")

    with open(config_path) as f:
        cfg = json.load(f)
    with open(scaler_pkl, "rb") as f:
        scaler_dict = pickle.load(f)

    loader, loaded = rebuild_split_loader(cfg, args.split, scaler_dict, args.batch_size)

    # Rebuild model from config.
    vol_indices = [loaded.feature_cols.index(c) for c in cfg["vol_feature_cols"]]
    mcfg = cfg["model"]
    model = VolTransformer(
        input_dim=len(loaded.feature_cols),
        vol_indices=vol_indices,
        d_model=mcfg["d_model"], nhead=mcfg["nhead"], num_layers=mcfg["num_layers"],
        dim_feedforward=mcfg["dim_feedforward"], dropout=mcfg["dropout"],
        pos_encoding=mcfg["pos_encoding"],
        attn_diagonal_bias=mcfg.get("attn_diagonal_bias", 0.0),
        conv_kernel=mcfg.get("conv_kernel", 3),
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.to(device).eval()

    out_path = run_dir / f"tsne_{args.split}.png"
    save_tsne_plot(
        model=model,
        loader=loader,
        device=device,
        out_path=out_path,
        n_samples=args.n_samples,
        perplexity=args.perplexity,
        title=f"t-SNE of pooled embeddings  ({run_dir.name}, {args.split} set)",
    )


if __name__ == "__main__":
    main()


# Live Inference

In [ ]:
import argparse
import pickle
import time
from pathlib import Path
from typing import Optional, Tuple

import ccxt
import numpy as np
import pandas as pd
import torch
from torch import nn
from datetime import datetime

from dataloader import load_data
from transformer import TimeSeriesTransformer


def find_latest_version(base: str = "outputs") -> Path:
    """Return the outputs/vN folder with the highest N."""
    base_path = Path(base)
    dirs = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith("v")]
    if not dirs:
        raise FileNotFoundError(f"No versioned run folders found in '{base}'.")
    versions = []
    for d in dirs:
        try:
            versions.append((int(d.name[1:]), d))
        except ValueError:
            pass
    versions.sort(key=lambda x: x[0])
    return versions[-1][1]


def get_next_data_dir(base: str = "data") -> Path:
    """Create data/data_1, data/data_2, ... and return the new folder."""
    base_path = Path(base)
    base_path.mkdir(exist_ok=True)
    existing = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith("data_")]
    versions = []
    for d in existing:
        try:
            versions.append(int(d.name.split("_")[1]))
        except (ValueError, IndexError):
            pass
    next_n = max(versions, default=0) + 1
    out_dir = base_path / f"data_{next_n}"
    out_dir.mkdir()
    return out_dir


def load_model_and_scaler(
    version_dir: Path,
    device: torch.device,
) -> Tuple[nn.Module, object]:
    """
    Load model weights and scaler from version_dir.
    input_dim is inferred from the scaler's n_features_in_.
    Reads config.json for encoding and attn_diagonal_bias if present.
    """
    import json

    ckpt_path = version_dir / "transformer_best.pt"
    scaler_path = version_dir / "scaler.pkl"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    if not scaler_path.exists():
        raise FileNotFoundError(f"Scaler not found: {scaler_path}")

    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)
    input_dim = scaler.n_features_in_

    config = {}
    config_path = version_dir / "config.json"
    if config_path.exists():
        with open(config_path) as f:
            config = json.load(f)

    model = TimeSeriesTransformer(
        input_dim=input_dim,
        pos_encoding=config.get("encoding", "sinusoidal"),
        attn_diagonal_bias=config.get("attn_diagonal_bias", 0.0),
    ).to(device)
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, scaler


def predict_latest_direction(
    model: nn.Module,
    scaler: object,
    live_csv: str,
    seq_len: int,
    up_threshold: float = 0.6,
    down_threshold: float = 0.4,
) -> Tuple[float, int, str]:
    """
    Run inference on the most recent seq_len samples from live_csv (after
    applying the same clean.R-style processing and scaling).

    Returns (prob_up, class_label, action), where:
      - prob_up: P(price in ~30s > now)
      - class_label: 1 for up, 0 for non-up
      - action: "LONG", "SHORT", or "FLAT" (no trade)
    """
    device = next(model.parameters()).device
    df, X, y, feature_cols = load_data(live_csv)
    if len(X) < seq_len:
        raise ValueError(f"Not enough rows after cleaning to form a window of length {seq_len}.")

    X_scaled = scaler.transform(X)
    X_window = X_scaled[-seq_len:]
    X_window_t = torch.from_numpy(X_window).float().unsqueeze(0).to(device)  # (1, T, F)

    with torch.no_grad():
        logits = model(X_window_t)
        prob_up = float(torch.sigmoid(logits).item())

    label = int(prob_up >= 0.5)

    if prob_up >= up_threshold:
        action = "LONG"
    elif prob_up <= down_threshold:
        action = "SHORT"
    else:
        action = "FLAT"

    return prob_up, label, action


def main() -> None:
    parser = argparse.ArgumentParser(description="Live-ish continuous inference for Transformer model")
    parser.add_argument(
        "--version-dir",
        type=str,
        default=None,
        help="Path to outputs/vN directory (default: auto-detect latest outputs/vN)",
    )
    parser.add_argument(
        "--live-csv",
        type=str,
        default=None,
        help="CSV path (default: data/data_N/live_orderflow.csv, new folder each run).",
    )
    parser.add_argument(
        "--data-base",
        type=str,
        default="data",
        help="Base folder for data/data_N when creating new run directories.",
    )
    parser.add_argument(
        "--seq-len",
        type=int,
        default=40,
        help="Sequence length used during training.",
    )
    parser.add_argument(
        "--up-threshold",
        type=float,
        default=0.6,
        help="Minimum prob(up) to go LONG.",
    )
    parser.add_argument(
        "--down-threshold",
        type=float,
        default=0.4,
        help="Maximum prob(up) to go SHORT.",
    )
    parser.add_argument(
        "--poll-interval",
        type=float,
        default=5.0,
        help="Seconds between checks for new data / trades.",
    )
    parser.add_argument(
        "--hold-seconds",
        type=float,
        default=30.0,
        help="Seconds to hold a position before closing (matches model prediction horizon).",
    )
    parser.add_argument(
        "--symbol",
        type=str,
        default="BTC/USDT",
        help="Trading pair symbol for live data (ccxt format).",
    )
    parser.add_argument(
        "--outputs-base",
        type=str,
        default="outputs",
        help="Base folder for outputs/vN when auto-detecting latest model.",
    )
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if args.version_dir is not None:
        version_dir = Path(args.version_dir)
    else:
        version_dir = find_latest_version(args.outputs_base)
    print(f"Using model from: {version_dir}")

    model, scaler = load_model_and_scaler(version_dir, device)

    # Create fresh data folder for this run (data/data_1, data/data_2, ...)
    if args.live_csv is not None:
        live_path = Path(args.live_csv)
        live_path.parent.mkdir(parents=True, exist_ok=True)
    else:
        data_dir = get_next_data_dir(args.data_base)
        live_path = data_dir / "live_orderflow.csv"
    print(f"Live CSV: {live_path}")

    # Live data exchange (BinanceUS, same as training data source)
    exchange = ccxt.binanceus()

    print("Starting continuous inference & trading loop. Press Ctrl+C to stop.\n")

    last_n_rows = 0
    position = "FLAT"  # "FLAT", "LONG", or "SHORT"
    entry_price: float | None = None
    entry_time: datetime | None = None
    total_pnl = 0.0
    trade_count = 0

    try:
        while True:
            # 1) Fetch a fresh live order book snapshot
            try:
                ob = exchange.fetch_order_book(args.symbol)
            except Exception as e:
                print(f"[error] fetch_order_book failed: {e}")
                time.sleep(args.poll_interval)
                continue

            # Compute the same raw features as the teacher CSV header
            bids = ob["bids"]
            asks = ob["asks"]
            if not bids or not asks:
                print("[warn] Empty order book, skipping this tick.")
                time.sleep(args.poll_interval)
                continue

            best_bid, bid_vol_1 = bids[0]
            best_ask, ask_vol_1 = asks[0]
            mid_price = (best_bid + best_ask) / 2.0
            spread = best_ask - best_bid
            fractional_price = mid_price - int(mid_price)
            micro_price = (best_bid * ask_vol_1 + best_ask * bid_vol_1) / (bid_vol_1 + ask_vol_1)

            # Depth-10 OBI metrics
            top_bids_10 = bids[:10]
            top_asks_10 = asks[:10]
            bid_vol_10 = sum(b[1] for b in top_bids_10)
            ask_vol_10 = sum(a[1] for a in top_asks_10)
            if (bid_vol_10 + ask_vol_10) == 0:
                obi_10 = 0.0
            else:
                obi_10 = (bid_vol_10 - ask_vol_10) / (bid_vol_10 + ask_vol_10)
            if (bid_vol_1 + ask_vol_1) == 0:
                obi_1 = 0.0
            else:
                obi_1 = (bid_vol_1 - ask_vol_1) / (bid_vol_1 + ask_vol_1)
            obi_diff = obi_10 - obi_1

            # Append a new row to the live CSV for this run
            header_needed = not live_path.exists()
            row_df = pd.DataFrame(
                [
                    {
                        "Time": datetime.now().strftime("%H:%M:%S"),
                        "Mid Price": float(mid_price),
                        "Micro Price": float(micro_price),
                        "fractional price": float(fractional_price),
                        "spread": float(spread),
                        "obi_1": float(obi_1),
                        "obi_10": float(obi_10),
                        "obi_diff": float(obi_diff),
                    }
                ]
            )
            row_df.to_csv(live_path, mode="a", index=False, header=header_needed)

            # 2) Reload cleaned data from the live CSV and act when enough rows exist
            df, X, y, feature_cols = load_data(str(live_path))
            n_rows = len(X)

            if n_rows < args.seq_len:
                print(f"[loop] rows={n_rows} (<{args.seq_len}) | waiting for more data...")
            else:
                current_price = float(mid_price)

                if n_rows > last_n_rows:
                    last_n_rows = n_rows
                    prob_up, label, action = predict_latest_direction(
                        model=model,
                        scaler=scaler,
                        live_csv=str(live_path),
                        seq_len=args.seq_len,
                        up_threshold=args.up_threshold,
                        down_threshold=args.down_threshold,
                    )
                    direction = "UP" if label == 1 else "DOWN/NO-UP"

                    # Simple position management:
                    # - If flat and strong signal → open position.
                    # - If in position and strong opposite signal → close, report PnL, and flip.
                    # - Otherwise hold.

                    msg_prefix = f"[rows={n_rows}] price={current_price:.2f} prob(up)={prob_up:.4f} → class: {direction} → signal: {action}"

                    if position == "FLAT":
                        if action == "LONG":
                            position = "LONG"
                            entry_price = current_price
                            entry_time = datetime.now()
                            print(f"{msg_prefix} | OPEN LONG @ {entry_price:.2f}")
                        elif action == "SHORT":
                            position = "SHORT"
                            entry_price = current_price
                            entry_time = datetime.now()
                            print(f"{msg_prefix} | OPEN SHORT @ {entry_price:.2f}")
                        else:
                            print(f"{msg_prefix} | NO POSITION")
                    else:
                        # Already in a trade
                        assert entry_price is not None
                        assert entry_time is not None
                        held_seconds = (datetime.now() - entry_time).total_seconds()
                        close_time_expired = held_seconds >= args.hold_seconds
                        close_and_flip = False

                        if position == "LONG" and action == "SHORT":
                            close_and_flip = True
                        elif position == "SHORT" and action == "LONG":
                            close_and_flip = True

                        if close_time_expired or close_and_flip:
                            # Close existing position
                            if position == "LONG":
                                trade_pnl = current_price - entry_price
                            else:  # SHORT
                                trade_pnl = entry_price - current_price

                            trade_ret_pct = (trade_pnl / entry_price) * 100.0
                            total_pnl += trade_pnl
                            trade_count += 1

                            reason = f"after {held_seconds:.0f}s" if close_time_expired else "signal flip"
                            print(
                                f"{msg_prefix} | CLOSE {position} @ {current_price:.2f} "
                                f"(entry {entry_price:.2f}) {reason} → PnL {trade_pnl:.2f} "
                                f"({trade_ret_pct:.2f}%), total PnL {total_pnl:.2f} "
                                f"over {trade_count} trades"
                            )

                            if close_and_flip:
                                # Flip into new position
                                position = "LONG" if action == "LONG" else "SHORT"
                                entry_price = current_price
                                entry_time = datetime.now()
                                print(f"    → OPEN {position} @ {entry_price:.2f}")
                            else:
                                # Time-based close: go flat
                                position = "FLAT"
                                entry_price = None
                                entry_time = None
                        else:
                            # Hold existing position
                            print(f"{msg_prefix} | HOLD {position} from {entry_price:.2f}")

                # If no new rows, still check time-based close and print heartbeat
                if n_rows == last_n_rows and n_rows >= args.seq_len:
                    # Close position if hold time expired (even without new inference)
                    if position != "FLAT" and entry_price is not None and entry_time is not None:
                        held_seconds = (datetime.now() - entry_time).total_seconds()
                        if held_seconds >= args.hold_seconds:
                            if position == "LONG":
                                trade_pnl = current_price - entry_price
                            else:
                                trade_pnl = entry_price - current_price
                            trade_ret_pct = (trade_pnl / entry_price) * 100.0
                            total_pnl += trade_pnl
                            trade_count += 1
                            print(
                                f"[loop] CLOSE {position} @ {current_price:.2f} (entry {entry_price:.2f}) "
                                f"after {held_seconds:.0f}s → PnL {trade_pnl:.2f} ({trade_ret_pct:.2f}%), "
                                f"total PnL {total_pnl:.2f} over {trade_count} trades"
                            )
                            position = "FLAT"
                            entry_price = None
                            entry_time = None

                    status = f"[loop] rows={n_rows} price={current_price:.2f} | position={position}"
                    if entry_price is not None and position != "FLAT":
                        # Mark-to-market PnL of the open position
                        if position == "LONG":
                            mtm = current_price - entry_price
                        else:
                            mtm = entry_price - current_price
                        status += f" | unrealized PnL={mtm:.2f}"
                    print(status)

            # Sleep before next check
            time.sleep(args.poll_interval)
    except KeyboardInterrupt:
        print("\nStopped continuous inference loop.")


if __name__ == "__main__":
    main()



usage: ipykernel_launcher.py [-h] [--version-dir VERSION_DIR]
                             [--live-csv LIVE_CSV] [--data-base DATA_BASE]
                             [--seq-len SEQ_LEN] [--up-threshold UP_THRESHOLD]
                             [--down-threshold DOWN_THRESHOLD]
                             [--poll-interval POLL_INTERVAL]
                             [--hold-seconds HOLD_SECONDS] [--symbol SYMBOL]
                             [--outputs-base OUTPUTS_BASE]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/spenser/Library/Jupyter/runtime/kernel-v32e546fff6bfa7ddbfd62db19208659543e0addea.json


SystemExit: 2

/opt/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
